<a href="https://colab.research.google.com/github/muhmmadabuzar-svg/Dark-Pattern-Thesis/blob/main/06_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import os

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Your completed Batch 2 file with human validation filled
input_file = "/content/drive/MyDrive/Dark Pattern Thesis/Batch 2/batch2_rule_based_outputs/batch2_rule_based_results_validation_ready.xlsx"

# Output folder
output_folder = "/content/drive/MyDrive/Dark Pattern Thesis/Batch 2/batch2_evaluation_outputs"
os.makedirs(output_folder, exist_ok=True)

# Output file
output_file = os.path.join(output_folder, "batch2_evaluation_results.xlsx")

print("Input file exists:", os.path.exists(input_file))
print("Output will be saved to:", output_file)

Input file exists: True
Output will be saved to: /content/drive/MyDrive/Dark Pattern Thesis/Batch 2/batch2_evaluation_outputs/batch2_evaluation_results.xlsx


In [ ]:
df = pd.read_excel(input_file)

print("Rows loaded:", len(df))
print("Columns:")
print(df.columns.tolist())

Rows loaded: 60
Columns:
['Screenshot_ID', 'Website', 'OCR_Text', 'OCR_Quality', 'Urgency Keywords', 'Misdirection Keywords', 'Obstruction Keywords', 'Urgency Detected', 'Misdirection Detected', 'Obstruction Detected', 'Combined_Text', 'Countdown Timer', 'Low Stock Text', 'Limited Time Badge', 'Urgency Emphasis', 'Discount Banner', 'Crossed-Out Price', 'Price Anchor', 'Dominant Buy Button', 'Sale Colour Emphasis', 'Popup Present', 'Popup Blocks Content', 'Forced Signup', 'Visual Fake Urgency Detected', 'Visual Misdirection Detected', 'Visual Obstruction Detected', 'Rule-Based Candidate Pattern', 'Rule Explanation', 'Rule Confidence', 'Human Fake Urgency', 'Human Visual Misdirection', 'Human Obstruction', 'Validation Notes', 'Structured Final Label']


In [ ]:
def clean_label(value):
    """
    Converts labels into clean Yes / No / Unclear values.
    """
    value = str(value).strip().lower()

    if value == "yes":
        return "Yes"
    elif value == "no":
        return "No"
    elif value == "unclear":
        return "Unclear"
    else:
        return "Unclear"

In [ ]:
def rule_contains(pattern, category):
    """
    Checks if a category appears in the rule-based candidate pattern.
    """
    pattern = str(pattern).lower()

    if category in pattern:
        return "Yes"
    else:
        return "No"


df["Rule Fake Urgency"] = df["Rule-Based Candidate Pattern"].apply(
    lambda x: rule_contains(x, "candidate_fake_urgency")
)

df["Rule Visual Misdirection"] = df["Rule-Based Candidate Pattern"].apply(
    lambda x: rule_contains(x, "candidate_visual_misdirection")
)

df["Rule Obstruction"] = df["Rule-Based Candidate Pattern"].apply(
    lambda x: rule_contains(x, "candidate_obstruction")
)

In [ ]:
df["Human Fake Urgency"] = df["Human Fake Urgency"].apply(clean_label)
df["Human Visual Misdirection"] = df["Human Visual Misdirection"].apply(clean_label)
df["Human Obstruction"] = df["Human Obstruction"].apply(clean_label)

In [ ]:
def match_or_mismatch(rule_label, human_label):
    """
    Compares rule-based label with human validation label.
    If human label is unclear, it is marked as Unclear.
    """

    if human_label == "Unclear":
        return "Unclear"

    if rule_label == human_label:
        return "Match"
    else:
        return "Mismatch"


df["Fake Urgency Match"] = df.apply(
    lambda row: match_or_mismatch(row["Rule Fake Urgency"], row["Human Fake Urgency"]),
    axis=1
)

df["Visual Misdirection Match"] = df.apply(
    lambda row: match_or_mismatch(row["Rule Visual Misdirection"], row["Human Visual Misdirection"]),
    axis=1
)

df["Obstruction Match"] = df.apply(
    lambda row: match_or_mismatch(row["Rule Obstruction"], row["Human Obstruction"]),
    axis=1
)

In [ ]:
def calculate_match_count(row):
    """
    Counts how many categories matched.
    Unclear categories are not counted.
    """

    match_columns = [
        row["Fake Urgency Match"],
        row["Visual Misdirection Match"],
        row["Obstruction Match"]
    ]

    valid_results = [x for x in match_columns if x != "Unclear"]
    matches = valid_results.count("Match")

    return f"{matches}/{len(valid_results)}"


df["Match Count"] = df.apply(calculate_match_count, axis=1)

In [ ]:
def calculate_metrics(data, rule_col, human_col, category_name):
    """
    Calculates evaluation metrics for one category.
    Unclear human labels are excluded from metric calculation.
    """

    # Use only rows where human label is Yes or No
    valid_data = data[data[human_col].isin(["Yes", "No"])]

    rule_yes = valid_data[rule_col] == "Yes"
    rule_no = valid_data[rule_col] == "No"
    human_yes = valid_data[human_col] == "Yes"
    human_no = valid_data[human_col] == "No"

    tp = ((rule_yes) & (human_yes)).sum()
    fp = ((rule_yes) & (human_no)).sum()
    fn = ((rule_no) & (human_yes)).sum()
    tn = ((rule_no) & (human_no)).sum()

    total = tp + fp + fn + tn

    accuracy = (tp + tn) / total if total > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    unclear_count = (data[human_col] == "Unclear").sum()

    return {
        "Category": category_name,
        "Valid Cases": total,
        "Unclear Cases": unclear_count,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "Accuracy": round(accuracy * 100, 1),
        "Precision": round(precision * 100, 1),
        "Recall": round(recall * 100, 1),
        "F1-score": round(f1 * 100, 1)
    }

In [ ]:
metrics = []

metrics.append(
    calculate_metrics(
        df,
        "Rule Fake Urgency",
        "Human Fake Urgency",
        "Candidate Fake Urgency"
    )
)

metrics.append(
    calculate_metrics(
        df,
        "Rule Visual Misdirection",
        "Human Visual Misdirection",
        "Candidate Visual Misdirection"
    )
)

metrics.append(
    calculate_metrics(
        df,
        "Rule Obstruction",
        "Human Obstruction",
        "Candidate Obstruction"
    )
)

metrics_df = pd.DataFrame(metrics)

metrics_df

,Category,Valid Cases,Unclear Cases,TP,FP,FN,TN,Accuracy,Precision,Recall,F1-score
0,Candidate Fake Urgency,52,8,26,1,20,5,59.6,96.3,56.5,71.2
1,Candidate Visual Misdirection,56,4,40,16,0,0,71.4,71.4,100.0,83.3
2,Candidate Obstruction,60,0,1,7,0,52,88.3,12.5,100.0,22.2


In [ ]:
match_summary = df["Match Count"].value_counts().reset_index()
match_summary.columns = ["Match Count", "Number of Screenshots"]

match_summary["Percentage"] = round(
    match_summary["Number of Screenshots"] / len(df) * 100,
    1
)

match_summary

,Match Count,Number of Screenshots,Percentage
0,3/3,21,35.0
1,2/3,18,30.0
2,1/3,12,20.0
3,2/2,4,6.7
4,1/1,3,5.0
5,1/2,2,3.3


In [ ]:
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Evaluation Calculations", index=False)
    metrics_df.to_excel(writer, sheet_name="Evaluation Metrics", index=False)
    match_summary.to_excel(writer, sheet_name="Row-Level Match Summary", index=False)

print("Evaluation file saved:")
print(output_file)

Evaluation file saved:
/content/drive/MyDrive/Dark Pattern Thesis/Batch 2/batch2_evaluation_outputs/batch2_evaluation_results.xlsx
